# Evaporative fraction

## Read data

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import evaspa.io
from evaspa.ef import EFModel1

In [3]:
from abc import ABC, abstractmethod
from dataclasses import InitVar, dataclass, field
from typing import ClassVar


@dataclass
class EFModel(ABC):
    """Abstract class for evaporative fraction model"""

    var: InitVar[np.ndarray]
    lst: InitVar[np.ndarray]
    dry_coeffs: tuple[float, float] = field(init=False)
    wet_coeffs: tuple[float, float] = field(init=False)

    def __post_init__(self, var: np.ndarray, lst: np.ndarray) -> None:
        """
        Post initilization method
        """
        self._estimate_edges(var, lst)

    @abstractmethod
    def _estimate_edges(self, var: np.ndarray, lst: np.ndarray) -> None:
        """
        Method to compute dry and wet edges
        """

    def dry_edge(self, var: float) -> float:
        """
        Compute dry temperature
        """
        return self.dry_coeffs[0] * var + self.dry_coeffs[1]

    def wet_edge(self, var: float) -> float:
        """
        Compute dry temperature
        """
        return self.wet_coeffs[0] * var + self.wet_coeffs[1]

    def compute_ef(self, var: np.ndarray, lst: np.ndarray) -> np.ndarray:
        """
        Method to compute evaporative fraction
        """
        ef = (self.dry_coeffs[0] * var + self.dry_coeffs[1] - lst) / (
            (self.dry_coeffs[0] - self.wet_coeffs[0]) * var
            + (self.dry_coeffs[1] - self.wet_coeffs[1])
        )
        ef = np.where(ef > 1, 1, ef)
        ef = np.where(ef < 0, 0, ef)
        return ef


@dataclass
class EFModel1(EFModel):
    """Abstract class for evaporative fraction model"""

    name: ClassVar[str] = "EF_1"
    doi: ClassVar[str] = "https://doi.org/10.1016/j.proenv.2013.06.035"

    nb_intervals: int = field(default=20, init=True)
    percentile: int = field(default=5, init=True)

    def _estimate_edges(self, var: np.ndarray, lst: np.ndarray) -> None:
        """
        Description
        -----------
        Domain division: Nb intervals of same pixels density

        Dry edge calculation:
        For each interval, compute the dry point:
          - compute the median of var values
          - compute the median of lst values of the percentile superior of the interval.
        Perform a linear regression from the dry points

        Wet edge calculation:
        For each interval, compute the wet point:
          - compute the median of var values
          - compute the median of lst values of the percentile inferior of the interval.
        Perform a linear regression from the wet points

        Outliers are not removed.

        Parameters
        ----------
        lst : np.array
            Land surface temperature
        var : np.array
            Variable used versus temperature (Albedo)
        """
        # removing common nan
        df = (
            pd.DataFrame(data={"lst": lst.reshape(-1), "var": var.reshape(-1)})
            .dropna(axis=0, how="any")
            .sort_values(by="var")
            .reset_index(drop=True)
        )

        var_values = []
        lst_sup_values = []
        lst_inf_values = []
        interval_size = int(np.floor(len(df) / self.nb_intervals))
        for i, group in df.groupby(df.index // interval_size):
            if i == self.nb_intervals:
                # Skip last not complete group
                break
            var_values.append(group["var"].median())
            lst_inf_values.append(
                group["lst"][
                    group["lst"] < np.percentile(group["lst"], self.percentile)
                ].median()
            )
            lst_sup_values.append(
                group["lst"][
                    group["lst"] > np.percentile(group["lst"], 100 - self.percentile)
                ].median()
            )

        self.dry_coeffs = np.polyfit(var_values, lst_sup_values, 1)
        self.wet_coeffs = np.polyfit(var_values, lst_inf_values, 1)

    def to_json(self) -> dict:
        """
        Return an dictionary
        """
        return {
            "name": self.name,
            "parameters": {
                "nb_intervals": self.nb_intervals,
                "percentile": self.percentile,
            },
        }

    def __repr__(self) -> str:
        """
        Print
        """
        return f"Method {self.name} with parameters: nb_intervals = {self.nb_intervals}, percentile = {self.percentile}"

In [4]:
dir_path = os.environ["EVASPA_TEST_DATA_PATH"]
file_path = os.path.join(dir_path,"Landsat_20230327_28PCA.tif")

In [5]:
data = evaspa.io.read_data(file_path)

In [ ]:
data

In [7]:
# Extract data
albedo = data["albedo"].data.flatten()
lst = data["lst"].data.flatten()
# mask values
lst_masked = np.where(albedo>0.13, lst, np.nan)
lst_masked = np.where(albedo<0.35, lst_masked, np.nan)

In [8]:
model = EFModel1(albedo,lst_masked,nb_intervals=20,percentile=0.5)

In [ ]:
# dimensions
# gridspec inside gridspec
fig = plt.figure(constrained_layout=True, figsize=(7, 5))

ax = plt.gca()

# plot all points
ax.scatter(albedo, lst, s=20, alpha=1, color='moccasin', clip_on=False)

# edges coordinates
a_max = np.round(np.ceil(np.nanmax(albedo)*10)/10,decimals=1)
a_min = np.round(np.floor(np.nanmin(albedo)*10)/10,decimals=1)
td_min =  model.dry_edge(a_min)
td_max =  model.dry_edge(a_max)
tw_min =  model.wet_edge(a_min)
tw_max =  model.wet_edge(a_max)

# plot edges
ax.plot([a_min, a_max], [td_min, td_max], color='tab:red', linewidth=3, label='dry edge', zorder=1)
ax.plot([a_min, a_max], [tw_min, tw_max], color='tab:blue', linewidth=3, label='wet edge', zorder=1)

# plot trapeze
ax.plot([a_min, a_min], [tw_min, td_min], c='k')
ax.plot([a_max, a_max], [tw_max, td_max], c='k')
ax.plot([a_min, a_max], [td_min, td_max], c='k')
ax.plot([a_min, a_max], [tw_min, tw_max], c='k')

#
ax.spines[["top", "right"]].set_visible(False)
ax.set_xlabel("Albedo")
ax.set_ylabel("Land Surface Temperature K")

# EF
alb = 0.35
ax.annotate('$EF=0$', xy=(alb, model.dry_edge(alb)+10), va='center', ha='right',fontsize=15)
ax.annotate('$EF=1$', xy=(alb, model.wet_edge(alb)-10), va='center', ha='right',fontsize=15)

# lettres
alb = 0.25
pt_a = model.dry_edge(alb)
pt_b = model.wet_edge(alb)
pt_c = pt_b + 0.6 * (pt_a-pt_b)
ax.annotate('A', xy=(alb-0.05, pt_a), va='center', ha='left', weight='bold',fontsize=12)
ax.annotate('C', xy=(alb-0.05, pt_c), va='center', ha='left', weight='bold',fontsize=12)
ax.annotate('B', xy=(alb-0.05, pt_b), va='center', ha='left', weight='bold',fontsize=12)

# points
ax.scatter(alb, pt_a, color='k', s=60)
ax.scatter(alb, pt_c, color='k', s=60)
ax.scatter(alb, pt_b, color='k', s=60)

# droite
ax.plot([alb, alb], [pt_a, pt_b], c='k')

ax.annotate(r"$EF=\dfrac{AC}{AB}$", xy=(.4, 300),fontsize=15)

ax.legend(fontsize=14)

# title
ax.set_title("Evaporative fraction method", fontsize=14)

# savefig
destfile = '../report/ef1.png'
fig.savefig(destfile)

plt.show()

In [10]:
# Extract data
lai = data["lai"].data.flatten()
fcover = 1-np.exp(-0.5*lai)
lst = data["lst"].data.flatten()
# mask values
lst_masked = np.where(fcover>0, lst, np.nan)
model2 = EFModel1(fcover,lst_masked,nb_intervals=20,percentile=0.05)

In [ ]:
# dimensions
# gridspec inside gridspec
fig = plt.figure(constrained_layout=True, figsize=(6, 5))

ax = plt.gca()

# plot all points
ax.scatter(fcover, lst, s=20, alpha=1, color='moccasin', clip_on=False)

# edges coordinates
fc_max = np.round(np.ceil(np.nanmax(fcover)*10)/10,decimals=1)
fc_min = np.round(np.floor(np.nanmin(fcover)*10)/10,decimals=1)
td_min =  model2.dry_edge(fc_min)
td_max =  model2.dry_edge(fc_max)
tw_min =  model2.wet_edge(fc_min)
tw_max =  model2.wet_edge(fc_max)

# plot edges
ax.plot([fc_min, fc_max], [td_min, td_max], color='tab:red', linewidth=3, label='dry edge', zorder=1)
ax.plot([fc_min, fc_max], [tw_min, tw_max], color='tab:blue', linewidth=3, label='wet edge', zorder=1)

# plot trapeze
ax.plot([fc_min, fc_min], [tw_min, td_min], c='k')
ax.plot([fc_max, fc_max], [tw_max, td_max], c='k')
ax.plot([fc_min, fc_max], [td_min, td_max], c='k')
ax.plot([fc_min, fc_max], [tw_min, tw_max], c='k')

#
ax.spines[["top", "right"]].set_visible(False)
ax.set_xlabel("Vegetation fraction cover",fontsize=12)
ax.set_ylabel("Land Surface Temperature K", fontsize=12)
ax.tick_params(axis='both', which='major', labelsize=11)

# EF
fc = 0.35
ax.annotate('Fully dry conditions\n$EF=0$', xy=(fc+0.01, model2.dry_edge(fc)+8), va='center', ha='right',fontsize=15)
ax.annotate('Fully wet conditions\n$EF=1$', xy=(fc, model2.wet_edge(fc)-8), va='center', ha='right',fontsize=15)
#ax.annotate('$EF=0$', xy=(fc, model2.dry_edge(fc)+5), va='center', ha='right',fontsize=15)
#ax.annotate('$EF=1$', xy=(fc, model2.wet_edge(fc)-5), va='center', ha='right',fontsize=15)

# 4 points
ax.annotate('(1)', xy=(fc_max+0.025, tw_max), va='center', ha='right', weight='bold', fontsize=11)
ax.annotate('well-watered \nvegetation', xy=(fc_max, tw_max-5), va='center', ha='right', fontsize=11)
ax.annotate('(2)', xy=(fc_max+0.025, td_max), va='center', ha='right', weight='bold', fontsize=11)
ax.annotate('water-stressed \nvegetation', xy=(fc_max, td_max+7), va='center', ha='right', fontsize=11)
ax.annotate('(3)', xy=(fc_min-0.02, td_min+2), va='center', ha='left', weight='bold', fontsize=11)
ax.annotate('dry bare \nsoil', xy=(fc_min+0.01, td_min+5), va='center', ha='left', fontsize=11)
ax.annotate('(4)', xy=(fc_min-0.02, tw_min-2), va='center', ha='left', weight='bold', fontsize=11)
ax.annotate('saturated \nbare soil', xy=(fc_min+0.01, tw_min-5), fontsize=11)

# lettres
fc = 0.25
pt_a = model2.dry_edge(fc)
pt_b = model2.wet_edge(fc)
pt_c = pt_b + 0.6 * (pt_a-pt_b)
ax.annotate('$T_{dry}$', xy=(fc-0.04, pt_a-2), va='center', ha='left', weight='bold',fontsize=14)
ax.annotate('$T$', xy=(fc-0.02, pt_c), va='center', ha='left', weight='bold',fontsize=14)
ax.annotate('$T_{wet}$', xy=(fc-0.04, pt_b+2), va='center', ha='left', weight='bold',fontsize=14)

# points
ax.scatter(fc, pt_a, color='k', s=60)
ax.scatter(fc, pt_c, color='k', s=60)
ax.scatter(fc, pt_b, color='k', s=60)

# droite
ax.plot([fc, fc], [pt_a, pt_b], c='k')

#ax.annotate("$EF=\dfrac{AC}{AB}$", xy=(.3, 340),fontsize=14)
ax.annotate(r"$EF=\dfrac{T_{dry}-T}{T_{dry}-T_{wet}}$", xy=(.2, 340),fontsize=18)

ax.legend(fontsize=10)
# title
ax.set_title("Evaporative fraction method", fontsize=18)

# savefig
destfile = '../report/ef2.png'
fig.savefig(destfile)

plt.show()




In [ ]:
# dimensions
# gridspec inside gridspec
fig = plt.figure(constrained_layout=True, figsize=(6, 5))

ax = plt.gca()

# plot all points
ax.scatter(fcover, lst, s=20, alpha=1, color='moccasin', clip_on=False)

# edges coordinates
fc_max = np.round(np.ceil(np.nanmax(fcover)*10)/10,decimals=1)
fc_min = np.round(np.floor(np.nanmin(fcover)*10)/10,decimals=1)
td_min =  model2.dry_edge(fc_min)
td_max =  model2.dry_edge(fc_max)
tw_min =  model2.wet_edge(fc_min)
tw_max =  model2.wet_edge(fc_max)

# plot edges
ax.plot([fc_min, fc_max], [td_min, td_max], color='tab:red', linewidth=3, label='dry edge', zorder=1)
ax.plot([fc_min, fc_max], [tw_min, tw_max], color='tab:blue', linewidth=3, label='wet edge', zorder=1)

# plot trapeze
ax.plot([fc_min, fc_min], [tw_min, td_min], c='k')
ax.plot([fc_max, fc_max], [tw_max, td_max], c='k')
ax.plot([fc_min, fc_max], [td_min, td_max], c='k')
ax.plot([fc_min, fc_max], [tw_min, tw_max], c='k')

#
ax.spines[["top", "right"]].set_visible(False)
ax.set_xlabel("Vegetation fraction cover")
ax.set_ylabel("Land Surface Temperature K")

# EF
fc = 0.35
ax.annotate('$EF=0$', xy=(fc, model2.dry_edge(fc)+5), va='center', ha='right',fontsize=15)
ax.annotate('$EF=1$', xy=(fc, model2.wet_edge(fc)-5), va='center', ha='right',fontsize=15)

# 4 points
ax.annotate('(1)', xy=(fc_max+0.02, tw_max), va='center', ha='right', weight='bold', fontsize=10)
ax.annotate('well-watered \nvegetation', xy=(fc_max, tw_max-5), va='center', ha='right', fontsize=10)
ax.annotate('(2)', xy=(fc_max+0.02, td_max), va='center', ha='right', weight='bold', fontsize=10)
ax.annotate('water-stressed \nvegetation', xy=(fc_max, td_max+7), va='center', ha='right', fontsize=10)
ax.annotate('(3)', xy=(fc_min-0.02, td_min), va='center', ha='left', weight='bold', fontsize=10)
ax.annotate('dry bare \nsoil', xy=(fc_min, td_min+5), va='center', ha='left', fontsize=10)
ax.annotate('(4)', xy=(fc_min-0.02, tw_min), va='center', ha='left', weight='bold', fontsize=10)
ax.annotate('saturated \nbare soil', xy=(fc_min, tw_min-5), fontsize=10)

# lettres
fc = 0.25
pt_a = model2.dry_edge(fc)
pt_b = model2.wet_edge(fc)
pt_c = pt_b + 0.6 * (pt_a-pt_b)
ax.annotate('A', xy=(fc-0.02, pt_a-2), va='center', ha='left', weight='bold',fontsize=12)
ax.annotate('C', xy=(fc-0.02, pt_c), va='center', ha='left', weight='bold',fontsize=12)
ax.annotate('B', xy=(fc-0.02, pt_b+2), va='center', ha='left', weight='bold',fontsize=12)

# points
ax.scatter(fc, pt_a, color='k', s=60)
ax.scatter(fc, pt_c, color='k', s=60)
ax.scatter(fc, pt_b, color='k', s=60)

# droite
ax.plot([fc, fc], [pt_a, pt_b], c='k')

ax.annotate(r"$EF=\dfrac{AC}{AB}$", xy=(.3, 340),fontsize=16)

ax.legend(fontsize=10)
# title
ax.set_title("Evaporative fraction method", fontsize=14)

# savefig
destfile = '../report/ef3.png'
fig.savefig(destfile)

plt.show()
